# CardioSense — Save the ECG models (same-kernel fast path)

**Run this at the end of `04_ecg_feature_fusion.ipynb`, in the same session.**

If that kernel has already died, use `04c_recover_ecg_from_checkpoints.ipynb`
instead — it rebuilds everything from the `.pt` files and costs a 1–3 minute
XGBoost re-run. This notebook is the version that costs nothing, because the
fitted objects are still in memory.

## What is and is not already saved

`train_joint()` writes `joint_seed{seed}.pt` every epoch, so the **gated joint
network and its signal-only head are safe on Drive** — weights, best-epoch
weights, feature count and feature names.

Three things are **not** in those files and are lost when this kernel dies:

| in memory now | why it matters |
|---|---|
| `xgb_models` | the five boosters behind `features_xgb` — the model §13 *selected* |
| `imputer`, `scaler` | without them the joint network cannot be fed anything |
| `experiments[...]["thresholds"]` | 0.5 is wrong for every PTB-XL superclass |

`features_xgb` is the one that beats the published baseline — macro F1 **0.771**
and macro AUC **0.9413** against 0.75 / 0.93 — and `summary.json` records only
its *metrics*, not the boosters.

## One detail that matters more than it looks

`features_xgb` was fitted on **`X_train_raw`**, the raw frame with NaNs intact,
because XGBoost learns a default direction for each missing value. Everything
else (the MLP, the joint network's feature branch) was fitted on the imputed,
scaled, clipped matrix.

Feeding scaled values to the boosters later raises **no error** — the medians sit
inside the range the trees already split on, so every prediction looks plausible
and is wrong. The bundle written below records `preprocessing="raw"` and
`ECGAdapter` refuses to load a bundle that does not declare a mode.

## 1. Confirm the objects are still in this kernel

In [ ]:
import numpy as np

required = {
    "xgb_models": "the five one-vs-rest boosters (§10)",
    "experiments": "per-model results, including tuned thresholds (§8 helpers)",
    "FINAL": "the validation-selected model name (§13)",
    "imputer": "SimpleImputer fitted on train (§8)",
    "scaler": "StandardScaler fitted on train (§8)",
    "FEATURE_NAMES": "training column order (§7b)",
    "CLASSES": "superclass names (§3)",
    "X_val_raw": "raw val frame, for the round-trip check (§7)",
    "y_val_feat": "val labels on the feature subset (§7)",
}
present = set(dir())
missing = [f"{k:14s} — {why}" for k, why in required.items() if k not in present]

if missing:
    raise RuntimeError(
        "These objects are not in this kernel:\n  " + "\n  ".join(missing)
        + "\n\nEither run 04_ecg_feature_fusion.ipynb to the end and re-run this "
          "cell in the same session, or — if that runtime is already gone — use "
          "04c_recover_ecg_from_checkpoints.ipynb instead."
    )

print(f"selected model (by val macro AUC): {FINAL}")
print(f"boosters: {len(xgb_models)}   classes: {list(CLASSES)}")
print(f"features: {len(FEATURE_NAMES)}")

## 2. Read the thresholds that `register()` already tuned

`register()` stored them under `experiments[name]["thresholds"]`, tuned on the
validation fold. They are not re-derived here — reusing the exact values keeps
this bundle consistent with the numbers notebook 04 reported.

In [ ]:
thr_xgb = np.asarray(experiments["features_xgb"]["thresholds"], dtype=float)
metrics_xgb = experiments["features_xgb"]["metrics"]

print("features_xgb thresholds (tuned on val):")
for cls, t in zip(CLASSES, thr_xgb):
    print(f"  {cls:5s} {t:.4f}")
print(f"\ntest macro F1  {metrics_xgb['macro_f1']:.4f}")
print(f"test macro AUC {metrics_xgb['macro_roc_auc']:.4f}")

if FINAL != "features_xgb":
    print(f"\n  NOTE: §13 selected '{FINAL}', not features_xgb. This notebook saves")
    print(f"  features_xgb regardless, because it is the recoverable one. If")
    print(f"  '{FINAL}' is a neural branch, 04c loads it from the .pt files.")

# The joint model's thresholds, if that branch ran.
thr_joint = (np.asarray(experiments["joint_gated"]["thresholds"], dtype=float)
             if "joint_gated" in experiments else None)
thr_signal = (np.asarray(experiments["signal_seresnet"]["thresholds"], dtype=float)
              if "signal_seresnet" in experiments else None)
print("\njoint_gated thresholds     :", "present" if thr_joint is not None else "absent")
print("signal_seresnet thresholds :", "present" if thr_signal is not None else "absent")

## 3. Write the bundles to Drive

In [ ]:
import json, pickle, time
from pathlib import Path

DRIVE_ECG = Path("/content/drive/MyDrive/CardioSense/models/ecg")
DRIVE_ECG.mkdir(parents=True, exist_ok=True)
CLIP_SD = 5.0

# -- the selected feature model. preprocessing="raw" is load-bearing.
feature_bundle = {
    "model": list(xgb_models),              # five one-vs-rest binary boosters
    "preprocessing": "raw",                 # NaNs intact; do NOT impute or scale
    "imputer": None,
    "scaler": None,
    "feature_cols": list(FEATURE_NAMES),
    "classes": list(CLASSES),
    "thresholds": thr_xgb,
    "clip_sd": CLIP_SD,
    "metadata": {
        "model_version": "ecg-features_xgb-v1",
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "selected_model": str(FINAL),
        "selected_by": "validation macro AUC (notebook 04 §13)",
        "n_features": int(len(FEATURE_NAMES)),
        "test_macro_f1": round(float(metrics_xgb["macro_f1"]), 4),
        "test_macro_auc": round(float(metrics_xgb["macro_roc_auc"]), 4),
        "thresholds_tuned_on": "validation fold (fold 9)",
        "is_calibrated": False,
        "note": ("Fitted on the raw feature frame with NaNs intact. Imputing or "
                 "scaling before predict_proba produces plausible wrong values "
                 "and raises nothing."),
    },
}
with open(DRIVE_ECG / "ecg_feature_model.pkl", "wb") as handle:
    pickle.dump(feature_bundle, handle)

# -- the preprocessor the joint network needs
preprocessor = {
    "imputer": imputer,
    "scaler": scaler,
    "feature_cols": list(FEATURE_NAMES),
    "clip_sd": CLIP_SD,
    "thresholds": thr_joint,
    "classes": list(CLASSES),
    "metadata": {
        "model_version": "ecg-joint_gated-v1",
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "n_features": int(len(FEATURE_NAMES)),
        "thresholds_tuned_on": "validation fold (fold 9)",
        "is_calibrated": False,
    },
}
with open(DRIVE_ECG / "ecg_feature_preprocessor.pkl", "wb") as handle:
    pickle.dump(preprocessor, handle)

written = ["ecg_feature_model.pkl", "ecg_feature_preprocessor.pkl"]

# -- thresholds for the waveform-only adapter, which needs no preprocessor
if thr_signal is not None:
    (DRIVE_ECG / "ecg_signal_thresholds.json").write_text(json.dumps({
        "classes": list(CLASSES),
        "thresholds": [round(float(t), 6) for t in thr_signal],
        "source": "experiments['signal_seresnet']['thresholds'] (notebook 04, val fold)",
    }, indent=2), encoding="utf-8")
    written.append("ecg_signal_thresholds.json")

for name in written:
    path = DRIVE_ECG / name
    print(f"  {name:34s} {path.stat().st_size / 1024:>8.1f} KB")

## 4. Verify each bundle reloads and reproduces its predictions

Saving a silently wrong artifact is worse than saving nothing. Each adapter is
constructed from the files on disk — not from the objects still in memory — and
checked against the in-memory probabilities on real validation records.

In [ ]:
import sys

SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from cardiosense.fusion import ECGAdapter, resolve_ecg_adapter

adapter = ECGAdapter(DRIVE_ECG)
assert adapter.available, adapter.load_error
assert adapter.preprocessing == "raw", adapter.preprocessing

norm_idx = list(CLASSES).index("NORM")
rows = [0, 1, 2, 3, 4]
worst = 0.0

for row in rows:
    raw_row = X_val_raw.iloc[row].to_dict()
    got = adapter.predict(raw_row)
    assert got.available, got.reason

    # In-memory reference: the boosters applied to the same raw row.
    matrix = X_val_raw.iloc[[row]].to_numpy(np.float32)
    reference = np.array([m.predict_proba(matrix)[0, 1] for m in xgb_models])
    want = float(1.0 - reference[norm_idx])

    worst = max(worst, abs(got.risk_score - want))
    print(f"  row {row}: adapter {got.risk_score:.6f}  in-memory {want:.6f}")

print(f"\nlargest difference over {len(rows)} rows: {worst:.2e}")
assert worst < 1e-6, "the reloaded bundle does not reproduce the in-memory model"
print("bundle verified")

In [ ]:
# A scaled matrix must give a DIFFERENT answer -- proof the mode is load-bearing
# rather than decorative, and that the adapter is honouring it.
row = 0
raw_matrix = X_val_raw.iloc[[row]].to_numpy(np.float32)
scaled_matrix = np.clip(
    scaler.transform(imputer.transform(X_val_raw.iloc[[row]])), -CLIP_SD, CLIP_SD
).astype(np.float32)

p_raw = np.array([m.predict_proba(raw_matrix)[0, 1] for m in xgb_models])
p_scaled = np.array([m.predict_proba(scaled_matrix)[0, 1] for m in xgb_models])

print("same boosters, same patient, two preprocessings:")
for cls, a, b in zip(CLASSES, p_raw, p_scaled):
    print(f"  {cls:5s} raw {a:.4f}   scaled {b:.4f}   diff {abs(a - b):.4f}")
print(f"\nmax difference {np.abs(p_raw - p_scaled).max():.4f} — and neither call errored.")
print("That gap is the bug the `preprocessing` field exists to prevent.")

In [ ]:
# What the fusion engine will pick up, and under which name.
chosen = resolve_ecg_adapter(
    DRIVE_ECG, search_roots=[Path("/content/drive/MyDrive/CardioSense")])
print("resolved adapter :", type(chosen).__name__, f"({chosen.variant})")
print("model_version    :", chosen.status()["model_version"])
print("calibrated       :", chosen.is_calibrated,
      f"-> reliability capped at {chosen.reliability:.2f}")

---

## Done

| file | unlocks |
|---|---|
| `ecg_feature_model.pkl` | `features_xgb` — the model §13 selected |
| `ecg_feature_preprocessor.pkl` | the joint waveform + feature adapter |
| `ecg_signal_thresholds.json` | tuned labels for the waveform-only adapter |

**Download copies to your own machine too.** Drive is one accidental deletion
from costing a retrain.

Nothing here is calibrated: every ECG adapter reports `is_calibrated=False` and
reliability capped at 0.50. That is correct and is what the fusion layer assumes
— the ECG contribution is a ranking score, and the report says so. Running
`cardiosense.ecg.calibrate` is what lifts the cap honestly.

Next: `05_multimodal_fusion.ipynb`.